# SFT / DPO Diagnostics

Run cells top to bottom. Cell 2 (load model) takes ~30s and the model stays in GPU memory across other cells, so you can iterate on prompts/positions without reloading.

**Switching adapter (SFT vs DPO)**: change `ADAPTER_PATH` in the setup cell, re-run cells 2 and beyond.

**Sections:**
1. Setup & load
2. Position diagnostic — what does the model predict at key positions in the chat template?
3. Generation comparison — base vs SFT/DPO on fixed test prompts
4. Training data inspection — verify what trl actually fed the model

## 1. Setup

In [ ]:
import json
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "Qwen/Qwen3-8B-Base"
ADAPTER_PATH = "checkpoints/sft"   # switch to "checkpoints/dpo" when DPO is done
REPO_ROOT = Path("/workspace/qwen3-sft-dpo-eval")

SYSTEM = "你是一个有用、诚实、无害的助手。"


def fmt(prompt: str, partial_answer: str = "") -> str:
    return (
        f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n{partial_answer}"
    )


def topk(model, tok, text: str, k: int = 5):
    ids = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**ids).logits[0, -1, :]
    probs = torch.softmax(logits, dim=0)
    top = torch.topk(probs, k)
    return [(int(i.item()), float(p.item()), tok.decode([i.item()])) for p, i in zip(top.values, top.indices)]


def show_topk(rows, label):
    print(f"  {label:<5s}: ", end="")
    for tok_id, p, s in rows:
        print(f"({s!r:14s} p={p:.3f})", end=" ")
    print()

In [ ]:
import os
os.chdir(REPO_ROOT)

tok = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
print(f"Adapter eos: {tok.eos_token!r}  id={tok.eos_token_id}")
print(f"Adapter pad: {tok.pad_token!r}  id={tok.pad_token_id}")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.bfloat16, device_map="cuda", trust_remote_code=True,
)
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()
print("\nModel ready. Trainable LoRA params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 2. Position diagnostic

At each position, compare what the LoRA-applied model predicts vs the base model (LoRA disabled). Key signals:
- **Position 1 (start of user)**: should look natural for base, slightly more instruction-style for SFT
- **Position 2 (start of assistant)**: SFT should commit to '你好' / '您好' style strongly
- **Position 3 (middle of answer)**: SFT should produce assistant-style continuation
- **Position 4 (end of complete answer)**: ⭐ CRITICAL — should predict `<|im_end|>` (151645) or `<|endoftext|>` (151643) at high probability

In [ ]:
POSITIONS = [
    ("1. start of user turn",
     f"<|im_start|>system\n{SYSTEM}<|im_end|>\n<|im_start|>user\n"),
    ("2. start of assistant turn",
     fmt("你好", "")),
    ("3. middle of answer (after '我是')",
     fmt("你好", "你好,我是")),
    ("4. end of complete answer (after '?')",
     fmt("你好", "你好,我是 AI 助手,有什么可以帮你的吗?")),
]

for desc, text in POSITIONS:
    print(f"\n[{desc}]")
    sft_top = topk(model, tok, text, k=3)
    with model.disable_adapter():
        base_top = topk(model, tok, text, k=3)
    show_topk(sft_top, "SFT")
    show_topk(base_top, "BASE")

## 3. Generation comparison

Greedy decoding for reproducibility. Stops at either `<|im_end|>` or `<|endoftext|>`.

In [ ]:
TEST_PROMPTS = [
    "用三句话解释什么是过拟合。",
    "你能帮我写一封请假邮件吗?需要请明天一天假,理由是看病。",
    "为什么天空是蓝色的?",
    "把下面这句话翻译成英文:今天天气真好,适合出去散步。",
    "写一段 Python 代码,计算斐波那契数列前 10 项。",
]

im_end_id = tok.convert_tokens_to_ids("<|im_end|>")
eot_id = tok.convert_tokens_to_ids("<|endoftext|>")
EOS_IDS = [im_end_id, eot_id]


def generate(model, prompt: str, max_new: int = 300) -> str:
    text = fmt(prompt)
    ids = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **ids,
            max_new_tokens=max_new,
            do_sample=False,
            pad_token_id=eot_id,
            eos_token_id=EOS_IDS,
        )
    decoded = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=False)
    return decoded.split("<|im_end|>")[0].split("<|endoftext|>")[0].strip()


for i, p in enumerate(TEST_PROMPTS, 1):
    print("=" * 90)
    print(f"[{i}] {p}")
    print("\n--- ADAPTER (SFT or DPO depending on ADAPTER_PATH) ---")
    print(generate(model, p))
    print("\n--- BASE ---")
    with model.disable_adapter():
        print(generate(model, p))
    print()

## 4. Training data inspection

Verify what's at the end of each training sample. For SFT we expect last token to be `<|im_end|>` (151645). For DPO, chosen/rejected each end with `<|im_end|>`.

In [ ]:
def inspect_jsonl_tail(path: Path, n_show: int = 8):
    if not path.exists():
        print(f"{path.name}: NOT FOUND")
        return
    with path.open() as f:
        sample = json.loads(f.readline())
    print(f"\n{path.name}: keys = {list(sample.keys())}")
    for key in sample:
        if not isinstance(sample[key], str):
            continue
        text = sample[key]
        ids = tok(text, add_special_tokens=False)["input_ids"]
        print(f"  [{key}] len={len(text)} chars, {len(ids)} tokens")
        print(f"    last {n_show} ids: {ids[-n_show:]}")
        print(f"    decoded:        {[tok.decode([t]) for t in ids[-n_show:]]}")


inspect_jsonl_tail(REPO_ROOT / "data/processed/sft_train.jsonl")
inspect_jsonl_tail(REPO_ROOT / "data/processed/dpo_train.jsonl")

## 5. Quick re-test after DPO

Once DPO training is done (`checkpoints/dpo` exists), edit cell 1's `ADAPTER_PATH = "checkpoints/dpo"`, re-run section 1 (kernel will reload model with new adapter), then sections 2-3.

**The verdict:**
- If DPO position-4 top-1 is `<|im_end|>` or `<|endoftext|>` at p > 0.3 → DPO fixed the EOS issue, ship it
- If still flat → the bug is deeper than per-stage; need to investigate trl/loss masking